<a href="https://colab.research.google.com/github/FabioContrera/criando-agentes-de-ia-com-agno/blob/main/Aula%203/Aula_3_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 3.2 — Adicionando o agente Analista ao time

**Curso:** Agno: Criando agentes e sistemas multiagente

**Aula:** 3 — Times de IA: como fazer agentes trabalharem juntos

**Instrutor:** Fabio Contrera

---

## Onde estamos

Na **Aula 3.1** montamos o time mínimo do ScoutAI FC:
- Treinador como líder
- Olheiro como especialista de busca.

Limites:

1. **Não há especialista em análise de desempenho.**
2. **Não há memória de sessão.**

Esta aula resolve os dois:

1. Criamos o agente **Analista** e uma **tool customizada**
3. Adicionamos o Analista ao time
4. Demonstramos a **necessidade da memória**
5. Ativamos memória de sessão


## Setup

Adicionamos `plotly`.


In [ ]:
!pip install -U agno openai tavily-python wikipedia plotly

In [ ]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

---

## Passo 1 — Por que o Analista?

**Plano da implementação:**

| Função | Quando usar |
|---|---|
| `comparar_em_barras` | Comparação direta entre jogadores em uma métrica única (gols, assistências) |
| `evolucao_em_linhas` | Progressão ao longo do tempo (gols por temporada) |




---

## Passo 2 — Criando as tools customizadas

Tool customizada no Agno é uma função Python decorada com `@tool`. A `description` é o que o LLM lê para decidir quando chamar (então ela tem que ser **clara e específica**).


- Gera grádico de barras

In [ ]:
import plotly.graph_objects as go
from agno.tools import tool

@tool(
    name="comparar_em_barras",
    description=(
        "Gera gráfico de barras verticais comparando jogadores em uma métrica única. "
        "Use quando o usuário pedir comparação direta entre jogadores em um único atributo "
        "(ex: gols, assistências, finalizações). "
        "Retorna confirmação de que o gráfico foi exibido."
    ),
)
def comparar_em_barras(
    jogadores: list[str],
    valores: list[float],
    metrica: str,
    titulo: str,
) -> str:
    """Gera gráfico de barras interativo comparando jogadores."""
    fig = go.Figure(data=[go.Bar(x=jogadores, y=valores)])
    fig.update_layout(
        title=titulo,
        xaxis_title="Jogador",
        yaxis_title=metrica.capitalize(),
        template="plotly_white",
        height=400,
    )
    fig.show()
    return f"Gráfico de barras '{titulo}' exibido para o usuário."


- Gera gráfico de linhas

In [ ]:
@tool(
    name="evolucao_em_linhas",
    description=(
        "Gera gráfico de linhas mostrando a evolução de jogadores ao longo do tempo. "
        "Use quando o usuário pedir progressão temporal "
        "(ex: 'como evoluiu nas últimas temporadas', 'desempenho ao longo dos anos'). "
        "Cada jogador é uma linha separada, períodos no eixo X. "
        "Retorna confirmação de que o gráfico foi exibido."
    ),
)
def evolucao_em_linhas(
    jogadores: list[str],
    periodos: list[str],
    valores_por_jogador: list[list[float]],
    metrica: str,
    titulo: str,
) -> str:
    """Gera gráfico de linhas interativo mostrando evolução temporal."""
    fig = go.Figure()
    for nome, valores in zip(jogadores, valores_por_jogador):
        fig.add_trace(go.Scatter(x=periodos, y=valores, mode="lines+markers", name=nome))
    fig.update_layout(
        title=titulo,
        xaxis_title="Período",
        yaxis_title=metrica.capitalize(),
        template="plotly_white",
        height=400,
    )
    fig.show()
    return f"Gráfico de linhas '{titulo}' exibido para o usuário."


---

## Passo 3 — Criando o Analista



In [ ]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat

modelo_analista = OpenAIChat(id="gpt-5.4")

analista = Agent(
    name="Analista",
    role="Analista de desempenho que interpreta dados e gera visualizações táticas",
    model=modelo_analista,
    instructions=[
        "Você é o Analista de desempenho do ScoutAI FC.",
        "Sua função é receber dados (geralmente do Olheiro) e produzir análise visual.",
        "Use `comparar_em_barras` para comparações diretas entre jogadores em uma métrica.",
        "Use `evolucao_em_linhas` para progressões ao longo do tempo.",
        "Sempre que gerar um gráfico, acompanhe de uma análise interpretativa de 2-3 frases "
        "destacando o que o gráfico revela.",
        "Quando receber pedido de gráfico ou visualização, "
        "SEMPRE chame `comparar_em_barras` ou `evolucao_em_linhas` — nunca produza ASCII, "
        "Mermaid ou tabelas como substituto. As tools são o entregável correto.",
        "Se receber dados insuficientes ou ambíguos, peça esclarecimento em vez de inventar números.",
    ],
    tools=[comparar_em_barras, evolucao_em_linhas],
    markdown=True,
)

---

## Passo 4 — Adicionando o Analista ao time

Agora montamos o time com **três agentes**:
- Treinador-líder (coordena e responde).
- Olheiro (busca),
- Analista (visualiza),



In [ ]:
from agno.tools.tavily import TavilyTools
from agno.tools.wikipedia import WikipediaTools

modelo_olheiro = OpenAIChat(id="gpt-5.4")

olheiro = Agent(
    name="Olheiro",
    role="Busca informação verificável sobre futebol em fontes externas (web e Wikipedia)",
    model=modelo_olheiro,
    instructions=[
        "Você é o Olheiro do ScoutAI FC. Sua função é buscar informação verificável em fontes externas.",

        "POLÍTICA DE FONTES — siga rigorosamente:",
        "• Para EVENTOS RECENTES (últimos jogos, convocações, lesões, forma atual): "
        "use Tavily (busca web) como primeira opção.",
        "• Para FATOS HISTÓRICOS CONSOLIDADOS (Copas antigas, biografias, técnicos do passado, "
        "regulamentos): use Wikipedia — é mais estruturada e citável.",
        "Em caso de dúvida, prefira Tavily.",
        "Retorne dados objetivos — números, datas, nomes, fatos. Não interprete nem opine.",
        "Se a busca falhar, diga claramente em vez de chutar.",
    ],
    tools=[TavilyTools(), WikipediaTools()],
    markdown=True,
)

In [ ]:
from agno.team import Team

modelo_treinador = OpenAIChat(id="gpt-5.4")

time_scoutai = Team(
    name="Treinador do ScoutAI FC",
    mode="coordinate",
    members=[olheiro, analista],
    model=modelo_treinador,
    instructions=[
        "Você é o Treinador do ScoutAI FC, líder da comissão técnica.",
        "Você coordena o Olheiro (busca dados) e o Analista (interpreta dados e gera gráficos).",

        "Quando o usuário pedir dados ou fatos verificáveis, delegue ao Olheiro.",
        "Quando o usuário pedir comparação visual, gráfico ou análise de evolução: "
        "primeiro garanta que o Olheiro trouxe os dados. Depois delegue ao Analista APENAS pedindo o gráfico",
        "o Analista tem tools próprias de Plotly e escolherá a apropriada.",
        "NÃO especifique formato (Mermaid, ASCII, tabela) — confie nas tools do Analista.",
        "Quando o usuário pedir conceito tático ou opinião, responda direto sem delegar.",
        "Sempre integre dados e análise antes de responder ao usuário, em português do Brasil.",
    ],
    markdown=True,
)

---

## Passo 5 — Time com Analista em ação



In [ ]:
time_scoutai.print_response(
    "Compare os gols pela Seleção dos jogadores Vinícius Júnior, Rodrygo e Raphinha."
    "Gere um gráfico para ilustrar essa comparação.",
    stream=True,
)

---

## Passo 6 — A dor da memória ausente

Vamos fazer **uma pergunta de follow-up**:

> *"E como esses jogadores evoluíram nos últimos 3 anos (2023, 2024 e 2025)?"*



In [ ]:
time_scoutai.print_response(
    "E como esses jogadores evoluíram nos últimos 3 anos (2023, 2024 e 2025)?",
    stream=True,
)

---

## Passo 7 — Ativando memória de sessão



In [ ]:
from agno.db.sqlite import SqliteDb

time_scoutai = Team(
    name="Treinador do ScoutAI FC",
    mode="coordinate",
    members=[olheiro, analista],
    model= modelo_treinador,
    instructions=[
        "Você é o Treinador do ScoutAI FC, líder da comissão técnica.",
        "Você coordena o Olheiro (busca dados) e o Analista (interpreta dados e gera gráficos).",
        "Quando o usuário pedir dados ou fatos verificáveis, delegue ao Olheiro.",
        "Quando o usuário pedir comparação visual, gráfico ou análise de evolução: "
        "primeiro garanta que o Olheiro trouxe os dados. Depois delegue ao Analista APENAS pedindo o gráfico",
        "o Analista tem tools próprias de Plotly e escolherá a apropriada.",
        "NÃO especifique formato (Mermaid, ASCII, tabela) — confie nas tools do Analista.",
        "Quando o usuário pedir conceito tático ou opinião, responda direto sem delegar.",
        "Sempre integre dados e análise antes de responder ao usuário, em português do Brasil.",
    ],
    db=SqliteDb(db_file="scoutai.db"),
    add_history_to_context=True,    # anexa histórico aos prompts
    num_history_runs=3,             # quantos turnos anteriores manter
    markdown=True,
)

### Repetindo a conversa, agora com memória


In [ ]:
# Pergunta 1 — comparação inicial
time_scoutai.print_response(
    "Compare os gols pela Seleção dos jogadores Vinícius Júnior, Rodrygo e Raphinha."
    "Gere um gráfico para ilustrar essa comparação.",
    stream=True,
)

In [ ]:
# Pergunta 2 — follow-up que depende de contexto
time_scoutai.print_response(
    "E como esses jogadores evoluíram nos últimos 3 anos (2023, 2024 e 2025)?",
    stream=True,
)

---

### Estado atual do produto

```
ScoutAI FC (estado atual)
│
├── ✅ Time funcional de 3 agentes
│   ├── Treinador (líder)
│   ├── Olheiro (busca: Tavily + Wikipedia)
│   └── Analista (visualização: barras e linhas)
├── ✅ Memória de sessão ativa
│
├── ❌ Sem acesso a base interna (RAG)            → Aula 3.3
├── ❌ Delegação ainda pode ser refinada          → Aula 3.3
└── ❌ Sem fluxo determinístico                   → Aula 4 (Workflows)
```

### Próxima aula

**Aula 3.3 — Refinando o time + RAG no Olheiro**

Vamos dar ao Olheiro acesso a uma **base interna de conhecimento** (relatórios fictícios da comissão técnica) usando RAG do Agno.
